# register-back-fn-after-wrap — ex2: register a binary op at TWO argnums and dispatch both

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `register-back-fn-after-wrap`. Running the final beacon cell reports progress against the `Backprop: register back fn` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: register back fn` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`register-back-fn-after-wrap`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "register-back-fn-after-wrap"
DD_SUBTOPIC = "Backprop: register back fn"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Register-back-fn-after-wrap — quick refresher

A tiny autograd needs to look up *which* backward fn corresponds to *which* forward op at *which* argument position. The convention is a dict keyed by `(forward_fn, argnum)`:

```python
class BackwardFuncLookup:
    def __init__(self): self._table = {}
    def add_back_func(self, fwd, argnum, back_fn):
        self._table[(fwd, argnum)] = back_fn
    def get_back_func(self, fwd, argnum):
        return self._table[(fwd, argnum)]

BACK_FUNCS = BackwardFuncLookup()
log = wrap_forward_fn(torch.log)
BACK_FUNCS.add_back_func(torch.log, 0, log_back)   # the wrap+register pair
```

Two steps, always in this order: (1) wrap the forward fn so it builds a Recipe; (2) register the backward fn so the reverse pass can find it. Binary ops register TWICE — once for argnum=0, once for argnum=1.

### Exercise 2 — register a binary op at TWO argnums and dispatch both

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the (fwd_fn, argnum) registration pattern to a BINARY op by registering two back fns — one per argument position — and dispatching each independently.
> Keywords: binary-op, argnum, multiply-back, argnum-dispatch
> ```

**KCs targeted:** `register-back-fn-after-wrap`, `arg-position-back-functions`

Implement `register_multiply(BACK_FUNCS)`. Given a fresh lookup table, register backward fns for `torch.multiply` at BOTH argnum=0 and argnum=1.

The backward fns are given to you:
- `multiply_back0(grad_out, out, x, y) -> grad_out * y`  (∂(xy)/∂x = y)
- `multiply_back1(grad_out, out, x, y) -> grad_out * x`  (∂(xy)/∂y = x)

**The key insight.** A binary op has TWO entries in the table — one for each input position. When backprop walks the graph for a node with `recipe.func == torch.multiply` and two parents at `recipe.parents = {0: x, 1: y}`, it dispatches:
- `get_back_func(torch.multiply, 0)(grad_out, out, x, y)` → grad for x
- `get_back_func(torch.multiply, 1)(grad_out, out, x, y)` → grad for y

Note both back fns receive ALL the forward args (`x` and `y`) even though each only differentiates w.r.t. one. The uniform calling convention again — the back fn picks what it needs.

The test then walks the dispatch loop for `multiply` over the parents dict and verifies both grads match the chain rule.

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]


def multiply_back0(grad_out, out, x, y):
    return grad_out * y


def multiply_back1(grad_out, out, x, y):
    return grad_out * x


def register_multiply(BACK_FUNCS: BackwardFuncLookup) -> None:
    """Register multiply_back0 at argnum=0 and multiply_back1 at argnum=1."""
    raise NotImplementedError()


def _test_ex2():
    BACK_FUNCS = BackwardFuncLookup()
    register_multiply(BACK_FUNCS)

    # Both argnums must be populated and point to the right fn.
    assert BACK_FUNCS.get_back_func(t.multiply, 0) is multiply_back0
    assert BACK_FUNCS.get_back_func(t.multiply, 1) is multiply_back1

    # --- dispatch loop simulating backprop for one multiply node ---
    x = t.tensor([2.0, 3.0, 4.0])
    y = t.tensor([5.0, 7.0, 11.0])
    out = x * y  # (10, 21, 44)
    grad_out = t.ones(3)

    # Pretend Recipe.parents = {0: x, 1: y}. Loop over (argnum, parent).
    parents = {0: x, 1: y}
    fwd_fn = t.multiply
    fwd_args = (x, y)
    grads = {}
    for argnum, parent in parents.items():
        back_fn = BACK_FUNCS.get_back_func(fwd_fn, argnum)
        grads[id(parent)] = back_fn(grad_out, out, *fwd_args)

    # d(x*y)/dx = y → grad_x should equal y.
    assert t.allclose(grads[id(x)], y), f'grad_x: {grads[id(x)]}'
    # d(x*y)/dy = x → grad_y should equal x.
    assert t.allclose(grads[id(y)], x), f'grad_y: {grads[id(y)]}'

    # Non-unit grad_out — chain rule should scale both grads.
    grad_out2 = t.tensor([2.0, 0.5, -1.0])
    g0 = BACK_FUNCS.get_back_func(t.multiply, 0)(grad_out2, out, x, y)
    g1 = BACK_FUNCS.get_back_func(t.multiply, 1)(grad_out2, out, x, y)
    assert t.allclose(g0, grad_out2 * y), f'g0: {g0}'
    assert t.allclose(g1, grad_out2 * x), f'g1: {g1}'

    # Witness vs autograd on z = (x*y).sum().
    x_ref = t.tensor([2.0, 3.0, 4.0], requires_grad=True)
    y_ref = t.tensor([5.0, 7.0, 11.0], requires_grad=True)
    (x_ref * y_ref).sum().backward()
    assert t.allclose(g0 if False else BACK_FUNCS.get_back_func(t.multiply, 0)(t.ones(3), x_ref.detach() * y_ref.detach(), x_ref.detach(), y_ref.detach()), x_ref.grad)
    assert t.allclose(BACK_FUNCS.get_back_func(t.multiply, 1)(t.ones(3), x_ref.detach() * y_ref.detach(), x_ref.detach(), y_ref.detach()), y_ref.grad)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        self._table = {}
    def add_back_func(self, fwd_fn, argnum, back_fn):
        self._table[(fwd_fn, argnum)] = back_fn
    def get_back_func(self, fwd_fn, argnum):
        return self._table[(fwd_fn, argnum)]


def multiply_back0(grad_out, out, x, y):
    return grad_out * y


def multiply_back1(grad_out, out, x, y):
    return grad_out * x


def register_multiply(BACK_FUNCS: BackwardFuncLookup) -> None:
    # Two entries — one per argument position.
    BACK_FUNCS.add_back_func(t.multiply, 0, multiply_back0)
    BACK_FUNCS.add_back_func(t.multiply, 1, multiply_back1)
```

**Why N entries for an N-ary op.** PyTorch's Function class has a single `backward` that returns a tuple of grads (one per input). ARENA's BACK_FUNCS table flattens that into N independent dict entries. Same information, different shape. The dict form falls out naturally from iterating `recipe.parents.items()` and dispatching per argnum.

**Why both back fns receive both args.** `multiply_back0` could be written `def multiply_back0(grad_out, out, x, y): return grad_out * y` or even `def multiply_back0(grad_out, out, _x, y): return grad_out * y`. The uniform `(grad_out, out, *original_args)` shape lets the dispatcher always call `back_fn(grad_out, out, *recipe.args, **recipe.kwargs)` — no per-fn unpacking logic.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()